# The Cloud Has a Footprint · AI series template

**How to use:** File → Save As → `dayNN_prompt_ai.ipynb` (e.g. `day04_clusters_ai.ipynb`), then edit the 5 numbered cells and Run All.

The idea of the series: data centers stay on every map, and each day adds **one** thing they press on (power, water, homes, heat, flood...).
This template is set up as **Day 1: every data center in the US + the giant AI campuses on the way**.

## 1 · Settings

In [ ]:
DAY      = 1
PROMPT   = "Points"
SLUG     = "points-ai"                       # short name for the files
NOTEBOOK = "_template_ai.ipynb"              # this file's name (for the code link in the caption)

TITLE    = "Where the Cloud Lives"
SUBTITLE = "Every data center in the US, and the giant AI campuses on the way"
STORY    = ("Small dots are the data centers running today. Solid circles are giant AI campuses already running. "
            "Glowing ones are still being built or expanding, sized by the power they will draw at full build.")
CREDITS  = ["im3", "epoch"]                  # data sources, see mapkit/style.py (CREDITS)

REGION   = "us48"     # "us48" | a metro: "nyc" "philly" "dc_nova" "sf_bay" "dallas" "austin" "seattle" "quincy_wa" | states: ["TX", "WA"]
TOOL     = "mapbox"   # "mapbox" (interactive, sponsor) | "folium" (interactive) | "static" (PNG only)
MAPBOX_STYLE = "dark" # "dark" | "light" | "standard" | "satellite"
PROJECTION   = "globe"  # mapbox only: "globe" | "mercator" | "albers" | "equalEarth"

## 2 · Load

In [ ]:
import mapkit as mk
from mapkit import ai, publish

dc  = ai.load_datacenters()     # existing data centers (IM3 / PNNL atlas)
big = ai.load_ai_campuses()     # frontier AI campuses, operating + coming (Epoch AI)

if REGION != "us48":
    dc, big = ai.clip(dc, REGION), ai.clip(big, REGION)
else:
    dc, big = ai.clip(dc, "us48"), ai.clip(big, "us48")

print(f"{len(dc):,} data centers · {len(big)} AI campuses")

## 3 · Analysis
The only part that really changes day to day. Add overlays here (power plants, water stress...) with `ai.attach`, `ai.count_near`, `ai.hexbin`, `ai.count_in`.

In [ ]:
# Numbers for the caption
top_states = dc["state_abb"].value_counts().head(5)
status     = big["status"].value_counts()
power_now  = big["power_now_mw"].sum()
power_peak = big["power_peak_mw"].sum()

print("Top states by data centers:", ", ".join(f"{s} {n}" for s, n in top_states.items()))
print("AI campuses by status:", ", ".join(f"{s} {n}" for s, n in status.items()))
print(f"AI campus power today: {power_now:,.0f} MW  ->  at full build: {power_peak:,.0f} MW "
      f"({power_peak / power_now:.1f}x)" if power_now else "")

## 4 · Map

In [ ]:
campus_popup = {"owner": "Owner", "status": "Status", "power_now_mw": "Power today (MW)",
                "power_peak_mw": "Power at full build (MW)", "peak_date": "Full build by"}
campus_fmt = {"power_now_mw": "{:,.0f}", "power_peak_mw": "{:,.0f}"}
running = big[big["status"] == "Operating"]
coming  = big[big["status"] != "Operating"]          # under construction + expanding

layers = [
    mk.Layer(dc, "Data centers today", color="#8fb3c9", radius=2.2, opacity=0.5,
             popup={"name": "Name", "operator": "Operator", "state_abb": "State"}),
    mk.Layer(running, "AI campuses running", color="#f2b134", size_column="power_peak_mw", radius=6,
             opacity=0.95, popup=campus_popup, fmt=campus_fmt),
    mk.Layer(coming, "AI campuses coming", color="#ff5c8a", size_column="power_peak_mw", radius=6,
             opacity=0.95, glow=True, popup=campus_popup, fmt=campus_fmt),
]
layers = [l for l in layers if len(l.gdf)]
bounds = ai.REGIONS[REGION] if isinstance(REGION, str) else None
kw = dict(day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=mk.style.credits(*CREDITS),
          series="ai", bounds=bounds)

if TOOL == "mapbox":
    m = mk.mapbox(layers, story=STORY, style=MAPBOX_STYLE, projection=PROJECTION, **kw)
elif TOOL == "folium":
    m = mk.interactive(layers, story=STORY, **kw)
else:
    m = mk.static(layers, crs="EPSG:5070", **kw)
m if TOOL == "folium" else None

## 5 · Export
Writes the map to `output/`, a thumbnail to `png_thumbnail/`, starts a caption in `captions/`, updates the gallery. Then commit + push in GitHub Desktop.

In [ ]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="ai",
             tool={"mapbox": "Python + Mapbox", "folium": "Python + Folium", "static": "Python"}[TOOL],
             notebook=NOTEBOOK, credits=mk.style.credits(*CREDITS, full=True))

# ArcGIS Pro day? uncomment:
# publish.to_arcgis({"datacenters": dc, "ai_campuses": big}, day=DAY)